# Stage 6: the final scoreboard

Every model, on **the same stocks, the same days, the same rulers**. This is the table the **How it works** page will show, so it has to be fair.

The contestants:
- **Naive**: next week = last week's wiggle
- **Naive-20**: next week = last month's wiggle (a tougher cheap yardstick)
- **GARCH** (scaled): the classic formula, fitted per stock
- **XGBoost**: the panel of advisors
- **LSTM**: the neural network that reads 40 days in order
- **Ensembles**: simple averages of the better models ("ask several friends, take the middle")

Kernel: **RiskWise (.venv)**.

In [ ]:
# Cell 1: load everything
import sys
sys.path.append("../pipeline")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

tab = pd.read_parquet("../data/features.parquet", columns=["sym", "date", "actual_next5", "naive5", "naive20"])
tab["date"] = pd.to_datetime(tab["date"])
EXAM_START = tab["date"].max() - pd.DateOffset(years=3)

xgb_p = pd.read_parquet("../data/preds_xgb.parquet")
lstm_p = pd.read_parquet("../data/preds_lstm.parquet")[["sym", "date", "lstm"]]
for d in (xgb_p, lstm_p):
    d["date"] = pd.to_datetime(d["date"])

garch = pd.concat([pd.read_csv(p, parse_dates=["date"]).assign(sym=p.stem) for p in Path("../data/garch").glob("*.csv")])
print("GARCH guesses:", f"{len(garch):,}", "rows for", garch["sym"].nunique(), "stocks")

## Making GARCH fair (the scale correction, again)

GARCH forecasts closing-price wiggle, while our target uses the whole day. As in Stage 4, we learn **one correction number per stock from the study period only** (dates before the exam, with a 10-day gap). Stocks with fewer than 150 study days cannot be corrected honestly, so GARCH skips them.

In [ ]:
# Cell 2: scale-correct GARCH, using study-period rows only
study = tab[tab["date"] <= EXAM_START - pd.Timedelta(days=10)].merge(garch, on=["sym", "date"])
ratio = (study["actual_next5"] / study["garch_raw"]).groupby(study["sym"])
scale = ratio.median()[ratio.size() >= 150]
print("stocks with a GARCH correction:", len(scale), "| typical correction:", round(scale.median(), 2))

g = garch[garch["sym"].isin(scale.index)].copy()
g["garch"] = g["garch_raw"] * g["sym"].map(scale)
g = g[["sym", "date", "garch"]]

In [ ]:
# Cell 3: one table with every model's guess on every exam day (only days where ALL models have a guess)
exam = tab[tab["date"] >= EXAM_START].dropna(subset=["actual_next5"])
t = (exam.merge(xgb_p[["sym", "date", "xgb", "block"]], on=["sym", "date"])
         .merge(lstm_p, on=["sym", "date"]).merge(g, on=["sym", "date"]))
t["ens_gx"] = (t["garch"] + t["xgb"]) / 2                    # GARCH + XGBoost
t["ens_xl"] = (t["xgb"] + t["lstm"]) / 2                     # XGBoost + LSTM
t["ens_all"] = (t["garch"] + t["xgb"] + t["lstm"]) / 3       # all three
print(f"{len(t):,} exam rows, {t['sym'].nunique()} stocks")

## Two rulers, so one cannot flatter a model by accident

1. **Miss share**: average miss divided by the stock's typical wiggle (the ruler we have used all along). Lower is better.
2. **QLIKE**: a standard ruler from volatility research. It punishes guessing *too low* more than too high, because underestimating risk is the costly mistake. Lower is better.

If the two rulers rank the models the same way, the ranking is trustworthy.

In [ ]:
# Cell 4: the scoreboard, both rulers
MODELS = {"Naive": "naive5", "Naive-20": "naive20", "GARCH": "garch", "XGBoost": "xgb", "LSTM": "lstm",
          "Ensemble: GARCH + XGBoost": "ens_gx", "Ensemble: XGBoost + LSTM": "ens_xl", "Ensemble: all three": "ens_all"}

def per_stock(df, col):
    miss = (df["actual_next5"] - df[col]).abs()
    r = (np.maximum(df["actual_next5"], 0.05) / np.maximum(df[col], 0.05)) ** 2    # floor at 0.05% a day so a flat week cannot give infinity
    qlike = r - np.log(r) - 1
    a = pd.DataFrame({"sym": df["sym"], "miss": miss, "qlike": qlike, "act": df["actual_next5"]}).groupby("sym")
    return a["miss"].mean() / a["act"].median(), a["qlike"].mean()

share, qlike = {}, {}
for name, col in MODELS.items():
    share[name], qlike[name] = per_stock(t, col)
share, qlike = pd.DataFrame(share), pd.DataFrame(qlike)

board = pd.DataFrame({
    "miss share": share.mean(),
    "gain vs Naive (%)": (1 - share.mean() / share["Naive"].mean()) * 100,
    "QLIKE": qlike.mean(),
    "stocks beating Naive (%)": (share.lt(share["Naive"], axis=0)).mean() * 100,
})
board.round(3)

How to read it: **gain vs Naive** is how much smaller the average miss is than Naive's. **QLIKE** is lower for better models.

**What happened when we ran it: the two rulers disagree, and that is the point of having two.**
- On **miss share**, XGBoost and LSTM are best (about 21% better than Naive), and GARCH is weak (under 2% better).
- On **QLIKE**, the ensembles that include GARCH are best, and GARCH alone beats Naive-20 and nearly matches XGBoost.
- Why: QLIKE punishes guessing *too low* much more than too high. GARCH's guesses are a little higher and smoother, so it is rarely far below a real storm.

For a **risk** app, underestimating risk is the costly mistake. So the ensemble of all three (GARCH + XGBoost + LSTM) is a sensible choice for the live app: it is still about 18.5% better than Naive on miss share, and it is the best or near-best on QLIKE. We do **not** tune the weights on the exam (that would flatter the result), so the average is a plain equal-weight one.

## Is the gain stable? Block by block

Six 6-month blocks of the exam. A model that only wins in one block should not be trusted.

In [ ]:
# Cell 5: miss share in each 6-month block
rows = []
for k, part in t.groupby("block"):
    row = {"block": k}
    for name, col in MODELS.items():
        row[name] = per_stock(part, col)[0].mean()
    rows.append(row)
pd.DataFrame(rows).set_index("block").round(3)

## Does it help everyone, or only some stocks?

We split the stocks two ways: by **how jumpy they are** (the stock's typical wiggle during the exam), and by **how much history they had before the exam**. If the winner only wins for calm, long-listed stocks, the app should say so.

In [ ]:
# Cell 6: breakdown by jumpiness and by history length
typical = t.groupby("sym")["actual_next5"].median()
jump = pd.qcut(typical, 3, labels=["calm third", "middle third", "jumpy third"])
pre_years = tab[tab["date"] < EXAM_START].groupby("sym").size() / 247
hist = pd.cut(pre_years.reindex(typical.index).fillna(0), [-0.01, 1, 5, 10, 100], labels=["under 1 year", "1 to 5 years", "5 to 10 years", "over 10 years"])

def group_table(groups):
    out = {}
    for label, members in groups.groupby(groups):
        out[label] = share.loc[members.index].mean()
    return pd.DataFrame(out).T

by_jump, by_hist = group_table(jump), group_table(hist)
print("By how jumpy the stock is (miss share, lower is better):")
display(by_jump.round(3)[["Naive", "Naive-20", "GARCH", "XGBoost", "LSTM", "Ensemble: all three"]])
print("\nBy years of history before the exam (stocks per group:", hist.value_counts().sort_index().to_dict(), ")")
display(by_hist.round(3)[["Naive", "Naive-20", "GARCH", "XGBoost", "LSTM", "Ensemble: all three"]])

## How sure are we about the gain? (a bootstrap)

The 1,500-odd stocks are like a sample. If we re-drew the sample many times, how much would "gain vs Naive" wobble? We resample the stocks 2,000 times and look at the range that covers 95% of the results.

**An honest caveat:** stocks move together (a market-wide shock hits many at once), so they are not fully independent, and this range is probably a little **too narrow**.

In [ ]:
# Cell 7: bootstrap the gain of the best ensemble against Naive
rng = np.random.default_rng(0)
best_name = board["miss share"].idxmin()
a, b = share["Naive"].values, share[best_name].values
gains = []
for _ in range(2000):
    pick = rng.integers(0, len(a), len(a))
    gains.append((1 - b[pick].mean() / a[pick].mean()) * 100)
low, high = np.percentile(gains, [2.5, 97.5])
print(f"best model: {best_name}")
print(f"gain vs Naive: {board.loc[best_name, 'gain vs Naive (%)']:.1f}%   (95% range from resampling stocks: {low:.1f}% to {high:.1f}%)")

In [ ]:
# Cell 8: chart of the scoreboard
ax = board["miss share"].sort_values(ascending=False).plot.barh(figsize=(8, 4), title="Average miss as a share of the typical wiggle (lower is better)")
ax.set_xlim(board["miss share"].min() * 0.9, board["miss share"].max() * 1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: save the scoreboard so the website's "How it works" page can show it
out = board.round(4).reset_index().rename(columns={"index": "model"})
out.to_csv("../data/scoreboard.csv", index=False)
long = []
for gtype, tbl in [("jumpiness", by_jump), ("history", by_hist)]:
    for grp, row in tbl.iterrows():
        for model, v in row.items():
            long.append({"group_type": gtype, "group": grp, "model": model, "miss_share": round(float(v), 4)})
pd.DataFrame(long).to_csv("../data/scoreboard_breakdown.csv", index=False)
pd.DataFrame(rows).round(4).to_csv("../data/scoreboard_blocks.csv", index=False)
t[["sym", "date", "block", "actual_next5", "naive5", "garch", "xgb", "lstm", "ens_all"]].to_parquet("../data/exam_table.parquet", index=False)
print("saved data/scoreboard.csv, scoreboard_breakdown.csv, scoreboard_blocks.csv, exam_table.parquet")
print("exam rows:", len(t), "| stocks:", t["sym"].nunique(), "| exam:", EXAM_START.date(), "to", t["date"].max().date())

## Reading the results honestly

- Look at which models beat the **cheap** yardsticks (Naive-20), not only Naive. Beating a weak opponent proves little.
- Compare each ensemble with its best single member. If the ensemble is only a hair better, the extra complexity may not be worth it for the live app.
- Notice who wins for the *jumpiest* third of stocks. Those are the ones where a wrong risk level matters most.
- No model here predicts surprises. The honest message for the website: these models capture the normal rhythm of calm and stormy spells, and they miss shocks.

## Your turn

1. Which single model is best? Which ensemble is best? Is the ensemble worth it?
2. In Cell 6, for which group does every model struggle most? Can you think why?
3. Explain back: **why do we show two different rulers (miss share and QLIKE) instead of one?**